# plotwave: getting started

This notebook is designed for new users.
It starts with a single audio example, then adds an envelope, a multi-audio plot, and finally a more annotated view.

## Setup

From the project root:

```bash
uv sync --all-groups
uv run jupyter lab
```

In [1]:
import numpy as np

import plotwave

## Example data

This notebook uses a short piano phrase as the source.
In this example, the model does two things: it generates a VHS-style synth version and it predicts the chord labels.


In [2]:
from pathlib import Path

import soundfile as sf

first_audio_path = Path("plotwave_C_piano.wav")
if not first_audio_path.exists():
    first_audio_path = Path("examples") / "plotwave_C_piano.wav"

second_audio_path = Path("plotwave_C_VHS.wav")
if not second_audio_path.exists():
    second_audio_path = Path("examples") / "plotwave_C_VHS.wav"


## 1. Audio only

This is the simplest `plotwave` call: one audio waveform.

In [3]:
plotwave.audio_file(first_audio_path, name="Piano").plot(
    layout={
        "title": {"text": "Piano source"},
        "height": 520,
    },
)


Plot(data=[AudioFileTrace(path=PosixPath('plotwave_C_piano.wav'), sr=48000.0, frames=400695, audio_format='wav', norm=False, clip=None, step=None, trace={'name': 'Piano'})], layout={'title': {'text': 'Piano source'}, 'height': 520}, config=None, points=None, display='auto', compress_audio=True, bitrate=None)

## 2. Audio + envelope

A common pattern is to keep the waveform and overlay a smoother envelope on top.

In [4]:
wav, sr = sf.read(first_audio_path, always_2d=False)
if np.ndim(wav) == 2:
    wav = wav.mean(axis=1)
window = 1_000
envelope = np.convolve(np.abs(wav), np.ones(window) / window, mode="same")

plotwave.plot(
    [
        plotwave.audio(wav, sr=sr, name="Piano"),
        plotwave.series(
            envelope,
            sr=sr,
            name="Envelope",
            fill="tozeroy",
        ),
    ],
    layout={
        "title": {"text": "Piano + envelope"},
        "height": 560,
    },
    points=3000
)


Plot(data=[AudioTrace(wav=array([ 5.77569008e-05,  5.12003899e-05,  4.32133675e-05, ...,
       -5.15867472e-02, -5.69313169e-02, -6.16967678e-02], shape=(400695,)), sr=48000.0, time=None, norm=False, clip=None, step=None, trace={'name': 'Piano'}), SeriesTrace(y=array([0.05856055, 0.05889025, 0.05924003, ..., 0.02481878, 0.02476982,
       0.02472672], shape=(400695,)), time=None, sr=48000.0, step=None, trace={'name': 'Envelope', 'fill': 'tozeroy'})], layout={'title': {'text': 'Piano + envelope'}, 'height': 560}, config=None, points=3000, display='auto', compress_audio=True, bitrate=None)

## 3. Two audio tracks

The dropdown next to `Play` lets you switch between the piano and the VHS-style synth.


In [5]:
plotwave.plot(
    [
        plotwave.audio_file(
            first_audio_path,
            name="Piano",
            color="#0f62fe",
            line={"width": 1.5},
        ),
        plotwave.audio_file(
            second_audio_path,
            name="VHS-style synth",
            color="#d62728",
            line={"width": 1.5, "dash": "dot"},
        ),
    ],
    layout={
        "title": {"text": "Piano and VHS-style synth"},
        "height": 560,
    },
)


Plot(data=[AudioFileTrace(path=PosixPath('plotwave_C_piano.wav'), sr=48000.0, frames=400695, audio_format='wav', norm=False, clip=None, step=None, trace={'name': 'Piano', 'color': '#0f62fe', 'line': {'width': 1.5}}), AudioFileTrace(path=PosixPath('plotwave_C_VHS.wav'), sr=48000.0, frames=400695, audio_format='wav', norm=False, clip=None, step=None, trace={'name': 'VHS-style synth', 'color': '#d62728', 'line': {'width': 1.5, 'dash': 'dot'}})], layout={'title': {'text': 'Piano and VHS-style synth'}, 'height': 560}, config=None, points=None, display='auto', compress_audio=True, bitrate=None)

## 4. Top and bottom segment lanes

This is a typical style transfer example. The model outputs a new VHS-style audio track and predicts the chords at the same time.
The top lane is the ground truth (labels). The bottom lane is the chord prediction.


In [6]:
import plotwave

plot = plotwave.plot(
    [
        plotwave.audio_file(first_audio_path, name="Piano"),
        plotwave.audio_file(second_audio_path, name="VHS-style synth"),
        plotwave.segments(
            [
                {"start": 0.0, "end": 2.09, "label": "C"},
                {"start": 2.09, "end": 4.18, "label": "Asus2"},
                {"start": 4.18, "end": 6.27, "label": "Esus2"},
                {"start": 6.27, "end": 8.3, "label": "B"},
            ],
            name="Ground Truth",
            lane="top",
            color_map={
                "C": "#0f62fe",
                "Asus2": "#367af7",
                "Esus2": "#82adfc",
                "B": "#bcd2fb",
            },
        ),
        plotwave.segments(
            [
                {"start": 0.0, "end": 2.09, "label": "C"},
                {"start": 2.09, "end": 4.18, "label": "D5"},
                {"start": 4.18, "end": 6.27, "label": "Esus2"},
                {"start": 6.27, "end": 8.3, "label": "B"},
            ],
            name="Prediction",
            lane="bottom",
            color_map={
                "C": "#0f62fe",
                "D5": "#ee9f0d",
                "Esus2": "#82adfc",
                "B": "#bcd2fb",
            },
        ),
    ]
)

plot

Plot(data=[AudioFileTrace(path=PosixPath('plotwave_C_piano.wav'), sr=48000.0, frames=400695, audio_format='wav', norm=False, clip=None, step=None, trace={'name': 'Piano'}), AudioFileTrace(path=PosixPath('plotwave_C_VHS.wav'), sr=48000.0, frames=400695, audio_format='wav', norm=False, clip=None, step=None, trace={'name': 'VHS-style synth'}), SegmentsTrace(items=[{'start': 0.0, 'end': 2.09, 'label': 'C', 'color': '#0f62fe'}, {'start': 2.09, 'end': 4.18, 'label': 'Asus2', 'color': '#367af7'}, {'start': 4.18, 'end': 6.27, 'label': 'Esus2', 'color': '#82adfc'}, {'start': 6.27, 'end': 8.3, 'label': 'B', 'color': '#bcd2fb'}], name='Ground Truth', lane='top', bg_alpha=0.08, box_alpha=0.92, textfont={}), SegmentsTrace(items=[{'start': 0.0, 'end': 2.09, 'label': 'C', 'color': '#0f62fe'}, {'start': 2.09, 'end': 4.18, 'label': 'D5', 'color': '#ee9f0d'}, {'start': 4.18, 'end': 6.27, 'label': 'Esus2', 'color': '#82adfc'}, {'start': 6.27, 'end': 8.3, 'label': 'B', 'color': '#bcd2fb'}], name='Prediction', lane='bottom', bg_alpha=0.08, box_alpha=0.92, textfont={})], layout=None, config=None, points=None, display='auto', compress_audio=True, bitrate=None)

# Export

In [7]:
plot.save("style_transfer.html")

PosixPath('style_transfer.html')